# 品詞の分布を調べる・比べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/05_corpus_statistics/pos_distribution.ipynb)

NLP クックブック 05-07 ／ コーパスの統計と可視化

テキストの品詞構成（名詞が多い、動詞が多い…）を調べて、文体やジャンルの違いを見たい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import io, re, urllib.request, zipfile
from collections import Counter
import pandas as pd
from fugashi import Tagger

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

tagger = Tagger()
base = "https://www.aozora.gr.jp/cards/"
works = {"吾輩は猫である": "000148/files/789_ruby_5639.zip", "坊っちゃん": "000148/files/752_ruby_2438.zip",
         "走れメロス": "000035/files/1567_ruby_4948.zip", "銀河鉄道の夜": "000081/files/456_ruby_145.zip"}

dist = {}
for title, path in works.items():
    c = Counter(w.feature.pos1 for line in aozora(base + path).split("\n") for w in tagger(line))
    total = sum(c.values())
    dist[title] = {pos: n / total for pos, n in c.items()}

df = pd.DataFrame(dist).fillna(0).sort_values("吾輩は猫である", ascending=False)
(df * 100).round(1)

,吾輩は猫である,坊っちゃん,走れメロス,銀河鉄道の夜
助詞,31.0,30.6,27.4,28.4
名詞,21.7,20.8,18.8,18.6
動詞,14.0,14.5,14.5,12.9
補助記号,10.6,10.7,17.8,11.7
助動詞,9.7,10.8,9.5,12.5
副詞,3.0,2.8,2.8,3.8
代名詞,2.2,2.6,3.0,2.5
形容詞,2.0,2.0,1.9,2.2
接尾辞,1.8,1.5,0.8,1.6
形状詞,1.3,1.2,0.9,2.0


In [3]:
# MVR (相の類 ÷ 用の類 × 100): 形容詞・副詞などが動詞に対して多いほど「ありさま描写的」(樺島・寿岳の文体指標)
mvr = (df.loc[["形容詞", "形状詞", "副詞", "連体詞"]].sum() / df.loc["動詞"] * 100).round(1)
print(mvr)

吾輩は猫である    51.8
坊っちゃん      47.2
走れメロス      45.0
銀河鉄道の夜     73.6
dtype: float64
